# 03 — Model comparison

Compare baseline (naive, seasonal naive), classical (ARIMA/ETS), Prophet (with holidays), and a simple ML model. Same train/val split; MAE and MAPE.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import pandas as pd
import numpy as np
from src.evaluation import train_val_split, mae, rmse, mape
data_path = root / "data" / "surgeries.csv"
if not data_path.exists():
    from src.data_generator import generate_surgery_counts
    generate_surgery_counts(output_path=str(data_path))
df = pd.read_csv(data_path, parse_dates=["date"])
# Single series: General Surgery @ H1
series = df[(df["specialty_name"] == "General Surgery") & (df["hospital_id"] == "H1")][["date", "surgery_count"]].sort_values("date").reset_index(drop=True)
train, val = train_val_split(series, date_col="date", val_days_or_ratio=0.2)
y_train = train["surgery_count"].values
y_val = val["surgery_count"].values
n_val = len(y_val)

## 1. Naive and seasonal naive

In [ ]:
# Naive: last value
pred_naive = np.full(n_val, y_train[-1])
# Seasonal naive: same weekday last week (period=7)
train_series = pd.Series(y_train, index=pd.RangeIndex(len(y_train)))
pred_seasonal = []
for i in range(n_val):
    # val day i corresponds to train end + i; same weekday = train end - 7 + (i % 7) approx
    idx = len(y_train) - 7 + (i % 7)
    pred_seasonal.append(y_train[idx] if idx >= 0 else y_train[0])
pred_seasonal = np.array(pred_seasonal)
print("Naive:        MAE =", round(mae(y_val, pred_naive), 2), " MAPE =", round(mape(y_val, pred_naive, skip_zeros=True), 1), "%")
print("Seasonal(7):  MAE =", round(mae(y_val, pred_seasonal), 2), " MAPE =", round(mape(y_val, pred_seasonal, skip_zeros=True), 1), "%")

## 2. ARIMA (statsmodels)

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
model_arima = ARIMA(y_train, order=(1, 0, 1), seasonal_order=(0, 0, 1, 7))
fit_arima = model_arima.fit()
pred_arima = fit_arima.forecast(steps=n_val)
print("ARIMA(1,0,1)(0,0,1,7): MAE =", round(mae(y_val, pred_arima), 2), " MAPE =", round(mape(y_val, pred_arima, skip_zeros=True), 1), "%")

## 3. Prophet (with holidays)

In [ ]:
from prophet import Prophet
train_prophet = train[["date", "surgery_count"]].rename(columns={"date": "ds", "surgery_count": "y"})
# Holidays for 2021–2023 (Thanksgiving = 4th Thu Nov)
holiday_dates = (
    [f"{y}-01-01" for y in range(2021, 2024)] +  # New Year
    [f"{y}-07-04" for y in range(2021, 2024)] +
    [f"{y}-12-25" for y in range(2021, 2024)] +
    [f"{y}-12-31" for y in range(2021, 2024)]
)
thanksgiving = []
for y in range(2021, 2024):
    # 4th Thursday of November
    nov1 = pd.Timestamp(f"{y}-11-01")
    thu = nov1 + pd.Timedelta(days=(3 - nov1.dayofweek) % 7 + 21)
    thanksgiving.append(thu.strftime("%Y-%m-%d"))
holidays_df = pd.DataFrame({
    "holiday": ["ny"] * 3 + ["july4"] * 3 + ["xmas"] * 3 + ["nye"] * 3 + ["thanksgiving"] * 3,
    "ds": pd.to_datetime(holiday_dates + thanksgiving),
})
m = Prophet(holidays=holidays_df, yearly_seasonality=True, weekly_seasonality=True)
m.fit(train_prophet)
future = m.make_future_dataframe(periods=n_val)
forecast = m.predict(future)
pred_prophet = forecast["yhat"].tail(n_val).values
print("Prophet+holidays: MAE =", round(mae(y_val, pred_prophet), 2), " MAPE =", round(mape(y_val, pred_prophet, skip_zeros=True), 1), "%")

## 4. Simple ML (lag + rolling + Ridge)

In [ ]:
from sklearn.linear_model import Ridge
full = pd.concat([train, val], ignore_index=True)
for L in [1, 7, 14]:
    full[f"lag_{L}"] = full["surgery_count"].shift(L)
full["roll7"] = full["surgery_count"].shift(1).rolling(7, min_periods=1).mean()
full["dow"] = full["date"].dt.dayofweek
full = full.dropna()
feat_cols = ["lag_1", "lag_7", "lag_14", "roll7", "dow"]
cut = val["date"].min()
X_train_ml = full.loc[full["date"] < cut, feat_cols]
y_train_ml = full.loc[full["date"] < cut, "surgery_count"]
X_val_ml = full.loc[full["date"] >= cut, feat_cols].head(n_val)
y_val_ml = full.loc[full["date"] >= cut, "surgery_count"].head(n_val).values
pred_ml = Ridge(alpha=1.0).fit(X_train_ml, y_train_ml).predict(X_val_ml)
if len(pred_ml) < n_val:
    pred_ml = np.r_[pred_ml, np.full(n_val - len(pred_ml), pred_ml[-1])]
print("Ridge (lags+roll+dow): MAE =", round(mae(y_val, pred_ml), 2), " MAPE =", round(mape(y_val, pred_ml, skip_zeros=True), 1), "%")

## Summary

Interpretability vs accuracy: naive/seasonal are baselines; ARIMA and Prophet capture seasonality and (for Prophet) holidays; ML can improve if features are informative. Choose by data size, need for holiday handling, and operational constraints.